# Swarm-Optimized Neural Network Ensembles
**Multi-Agent Systems — Final Project**

This notebook is a thin driver. All logic lives in the `.py` modules on Drive.
The only cell you should need to edit between runs is **Section 3 — Configuration**.

---
**Sections**
1. Environment Setup
2. GPU Check
3. Configuration ← edit this between runs
4. Smoke Test
5. Full Ablation Run
6. Results & Plots
7. Hyperparameter Sweep (optional)

## 1 — Environment Setup

In [ ]:
# Mount Google Drive — your project folder lives here
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

# ── Adjust this if your folder is in a different location on Drive ──
PROJECT_ROOT  = Path('/content/drive/MyDrive/Final_Project')

# CIFAR-10 downloads to Colab local storage — re-downloads each session (~60s)
# Keeps Drive clean. Change to a Drive path if you prefer to persist it.
DATA_ROOT     = Path('/content/data')

# Checkpoints saved to Drive — survive session death
CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / 'checkpoints'

# Add project root to Python path so all imports work
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root : {PROJECT_ROOT}')
print(f'Data root    : {DATA_ROOT}')
print(f'Checkpoints  : {CHECKPOINT_DIR}')
print(f'Project exists on Drive: {PROJECT_ROOT.exists()}')

In [ ]:
# Install dependencies
# torchvision and torch are pre-installed on Colab — this only adds wandb
%pip install -q wandb

In [ ]:
# Log in to W&B — paste your API key when prompted
# Get your key at: https://wandb.ai/settings
import wandb
wandb.login()

## 2 — GPU Check

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No GPU detected! Go to Runtime → Change runtime type → GPU (A100)'
)

gpu_name   = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU          : {gpu_name}')
print(f'Memory       : {gpu_memory:.1f} GB')
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA         : {torch.version.cuda}')

DEVICE = 'cuda'

## 3 — Configuration

**This is the only cell you need to edit between runs.**

Key knobs:
- `conditions` — which ablation cells to run (`None` = all 8)
- `epochs` — 50 is a good default; increase to 100 for a final run
- `alpha`, `beta`, `gamma` — rule strengths when active
- `k` — neighborhood size (3 = sparse, 9 = full connectivity for N=10)

In [ ]:
from experiments.ablation import AblationConfig

cfg = AblationConfig(
    # ── Swarm rule strengths ──────────────────────────────────────────
    alpha = 0.3,     # gradient alignment
    beta  = 0.5,     # separation  → equilibrium spacing = β/γ = 5.0
    gamma = 0.1,     # cohesion
    k     = 3,       # k-NN neighborhood size

    # ── Training ─────────────────────────────────────────────────────
    n_agents     = 10,
    epochs       = 50,
    batch_size   = 512,
    subset_size  = None,     # None = full CIFAR-10 (50k)
    device       = DEVICE,
    lr           = 1e-3,
    weight_decay = 1e-4,

    # ── Logging ───────────────────────────────────────────────────────
    cka_interval     = 5,       # compute CKA every N epochs
    landscape_at_end = True,    # loss landscape after training
    wandb_project    = 'swarm-optimization',
    wandb_mode       = 'online',

    # ── Conditions ───────────────────────────────────────────────────
    # None = run all 8. Or specify a subset:
    # conditions = ['baseline', 'separation', 'full_swarm']
    conditions = None,

    # ── Paths ─────────────────────────────────────────────────────────
    checkpoint_dir = CHECKPOINT_DIR,
)

print(f'Conditions   : {cfg.conditions or "all 8"}')
print(f'Agents       : {cfg.n_agents}')
print(f'Epochs       : {cfg.epochs}')
print(f'Batch size   : {cfg.batch_size}')
print(f'Dataset      : {"full CIFAR-10" if cfg.subset_size is None else f"{cfg.subset_size} samples"}')
print(f'Rules        : α={cfg.alpha}  β={cfg.beta}  γ={cfg.gamma}  k={cfg.k}')

## 4 — Smoke Test

Run 3 agents × 2 epochs × 2 conditions before committing to the full run.
This confirms the environment, Drive paths, and imports all work correctly.

**Expected output:** two conditions complete, summary table prints, no errors.

In [ ]:
from experiments.ablation import run_ablation, AblationConfig

smoke_cfg = AblationConfig(
    n_agents         = 3,
    epochs           = 2,
    subset_size      = 500,
    device           = DEVICE,
    cka_interval     = 1,
    landscape_at_end = True,    # must match full run — catches device issues early
    wandb_mode       = 'disabled',
    checkpoint_dir   = CHECKPOINT_DIR / 'smoke',
    conditions       = ['baseline', 'full_swarm'],
    # inherit rule strengths from main cfg
    alpha = cfg.alpha,
    beta  = cfg.beta,
    gamma = cfg.gamma,
)

smoke_results = run_ablation(smoke_cfg)
print('\n✓ Smoke test passed — safe to proceed with full run.')

## 5 — Full Ablation Run

Trains all configured conditions sequentially.
Metrics stream live to your W&B dashboard at:
**https://wandb.ai/osro6012-university-of-colorado-boulder/swarm-optimization**

Estimated time on A100 (50 epochs, 10 agents, full CIFAR-10): ~2-3 hours total.

In [ ]:
# Pass data_root so CIFAR-10 downloads to local Colab storage
# We patch the default at runtime to avoid editing the source files
import data.cifar as cifar_module
cifar_module._DEFAULT_DATA_ROOT = DATA_ROOT

results = run_ablation(cfg)

## 6 — Results & Plots

In [ ]:
# Summary comparison table
from experiments.ablation import compare_conditions
compare_conditions(results)

In [ ]:
import matplotlib.pyplot as plt
from visualization.plots import plot_diversity_curves, plot_training_curves

# ── Diversity curves: one plot per condition ──────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    if res['cka_history']:
        fig_cka = plot_diversity_curves(res['cka_history'])
        # Redraw onto subplot
        src_ax = fig_cka.axes[0]
        for line in src_ax.lines:
            ax.plot(line.get_xdata(), line.get_ydata(),
                    label=line.get_label(), linewidth=line.get_linewidth())
        ax.set_title(label, fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel('Mean CKA', fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.close(fig_cka)

fig.suptitle('Representational Diversity by Condition', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'diversity_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── Training loss curves side by side ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    epochs_  = [d['epoch']     for d in res['history']]
    means    = [d['mean_loss'] for d in res['history']]
    ax.plot(epochs_, means, linewidth=2, color='black', label='mean')
    for i in range(len(res['history'][0]['agent_losses'])):
        agent_curve = [d['agent_losses'][i] for d in res['history']]
        ax.plot(epochs_, agent_curve, linewidth=0.8, alpha=0.4)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Loss', fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Training Loss by Condition', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── Final CKA matrix for each condition at gap layer ──────────────────────
from visualization.plots import plot_cka_matrix

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    if res['cka_history']:
        last   = res['cka_history'][-1]
        matrix = last['gap']['matrix']
        fig_cka = plot_cka_matrix(matrix, layer='gap', epoch=last['epoch'])
        src_ax  = fig_cka.axes[0]
        img     = src_ax.images[0]
        ax.imshow(img.get_array(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
        ax.set_title(label, fontsize=10)
        ax.axis('off')
        plt.close(fig_cka)

fig.suptitle('Final CKA Similarity (GAP layer)', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'cka_matrices.png'), dpi=150)
plt.show()

## 7 — Hyperparameter Sweep (Optional)

Run this section only after the ablation has identified a winning condition.
The sweep fine-tunes the rule strengths for that condition using Bayesian optimization.

**Step 1** creates the sweep on W&B (run once).
**Step 2** launches the agent (can duplicate this cell for parallel agents).

In [ ]:
# ── Step 1: Create sweep — run ONCE, then comment out ────────────────────
from experiments.hyperparam_sweep import create_sweep

SWEEP_CONDITION = 'full_swarm'   # ← change to your winning condition

sweep_id = create_sweep(
    condition = SWEEP_CONDITION,
    project   = 'swarm-optimization',
    method    = 'bayes',
    n_trials  = 30,
)
print(f'sweep_id = {sweep_id!r}  ← paste into Step 2')

In [ ]:
# ── Step 2: Launch agent ──────────────────────────────────────────────────
from experiments.hyperparam_sweep import run_sweep_agent

SWEEP_ID = ''   # ← paste sweep_id from Step 1

run_sweep_agent(
    sweep_id       = SWEEP_ID,
    condition      = SWEEP_CONDITION,
    project        = 'swarm-optimization',
    n_agents       = cfg.n_agents,
    epochs         = cfg.epochs,
    subset_size    = cfg.subset_size,
    device         = DEVICE,
    checkpoint_dir = CHECKPOINT_DIR / 'sweep',
    count          = 10,   # runs per agent process
)